# Rag

## Basic (Standard)

In [1]:
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_ollama import OllamaEmbeddings
from langchain_classic.chains.retrieval_qa.base import RetrievalQA
from huggingface_hub import login
import os
from dotenv import load_dotenv
import subprocess

/home/vignesh/.nlpvenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_21273/2625501338.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


### Stop llm (ollama) and Read PDF Docs

In [2]:
from pathlib import Path
import logging
from tqdm import tqdm
# Tell pypdf to only print errors, which hides the font warnings
logging.getLogger("pypdf").setLevel(logging.ERROR)
def stop_ollama(model:str):
    """
    Stop the Ollama model if it is running.
    """
    try:
        # Check if the model is running
        result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
        if model in result.stdout:
            # Stop the model
            subprocess.run(["ollama", "stop", model])
            print(f"Stopped Ollama model: {model}")
        else:
            print(f"Ollama model {model} is not running.")
    except Exception as e:
        print(f"Error stopping Ollama model: {e}")



def get_pdf_docs_via_text_spliter(folder_path:Path):
    file_paths = [os.path.join(folder_path,file) for file in os.listdir(folder_path)]
    docs = []
    text_spliter = RecursiveCharacterTextSplitter(chunk_size=2046,chunk_overlap=1046)
    print(f"Total no of files in {folder_path}: {len(os.listdir(folder_path))}")
    
    for path in tqdm(file_paths, desc="Processing PDFs", leave=True):
        PDF_loader = PyPDFLoader(file_path=str(path))
        pdf_doc = PDF_loader.load()
        docs.extend(text_spliter.split_documents(pdf_doc))
    
    return docs
    

In [6]:
PAPERS = 'inputs/pdf'
docs = get_pdf_docs_via_text_spliter(PAPERS)


Total no of files in inputs/pdf: 20


Processing PDFs: 100%|██████████| 20/20 [00:42<00:00,  2.15s/it]


In [7]:
print(docs[0])

page_content='Quantitative Structure�Permittivity Relationship Study of a Series
of Polymers
Published as part of ACS Materials Au virtual special issue “2023 Rising Stars”.
Yevhenii Zhuravskyi, Kweeni Iduoku, Meade E. Erickson, Anas Karuth, Durbek Usmanov,
Gerardo Casanola-Martin, Maqsud N. Sayfiyev, Dilshod A. Ziyaev, Zulayho Smanova,
Alicja Mikolajczyk,* and Bakhtiyor Rasulev*
Cite This: ACS Mater. Au 2024, 4, 195−203
 Read Online
ACCESS
 Metrics & More
 Article Recommendations
ABSTRACT: Dielectric constant is an important property which
is widely utilized in many scientific fields and characterizes the
degree of polarization of substances under the external electric
field. In this work, a structure−property relationship of the
dielectric constants (ε) for a diverse set of polymers was
investigated. A transparent mechanistic model was developed
withtheapplicationofamachinelearningapproachthatcombines
geneticalgorithmandmultiplelinearregressionanalysis,toobtain
amechanisticallyexplai

In [8]:
print(type(docs))

<class 'list'>


### Vector DB (Chroma)

In [35]:

class Vector_DB_store:
    def __init__(self,docs:list = None,model:str = 'nomic-embed-text:latest',base_url:str = 'http://127.0.0.1:11434',vector_db_path:Path = r'vector_db/Chroma/polymer_model_papers'):
        self._model = model
        self._docs = docs
        self._persist_path = vector_db_path
        os.makedirs(self._persist_path,exist_ok=True)
        self._embed = OllamaEmbeddings(model=self._model,base_url=base_url)
        
    def save(self,batch_size=4):
        if self._docs ==None:
            raise ValueError(f"Error: docs must me passed to save, docs:{self._docs}")
        vector_db = self.load()
                
        for i in tqdm(range(0,len(self._docs),batch_size), desc="Generating Embeddings & Storing"):
                    batch = self._docs[i : i + batch_size]
                    vector_db.add_documents(batch)
        # Chroma.from_documents(
        #     self._docs,
        #     embedding=self._embed,
        #     persist_directory=self._persist_path,
        # )
        stop_ollama(self._model)
    
    def load(self,):
        return Chroma(
            persist_directory=self._persist_path,
            embedding_function=self._embed,
        )
        
    def get_retriver(self,k:int=4):
        vector_db = self.load()
        return vector_db.as_retriever(k=k)

In [37]:
# vector_db_store = Vector_DB_store(docs=docs)
vector_db_store = Vector_DB_store()

#### Store Vector DB

In [ ]:
vector_db_store.save()

In [38]:
retriver = vector_db_store.get_retriver(k=7)

/tmp/ipykernel_21273/891149308.py:25: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  return Chroma(


### Ollama Chat model (DeepSeek-r1)

In [4]:
from langchain_ollama import ChatOllama

In [66]:

OLLAMA_MODEL_NAME = 'deepseek-r1:latest'

ollama_model = ChatOllama(
    model=OLLAMA_MODEL_NAME,
    validate_model_on_init=True,
    num_ctx= 22921,
    num_predict=8046,
    stream=True
)

In [16]:
system_prompt = "You are a helpful assistant that can answer questions and return the reponse in markdown format."
message = [
    ("system",system_prompt),
    ("human","explain how vllm works, and how different from ollama and loading and using via hugging face transformers?")
]
for chunk in ollama_model.stream(message):
    print(chunk.text,end="", flush=True)


Here's a breakdown of how **vLLM**, **Ollama**, and **Hugging Face Transformers** work and their key differences:

---

### 🔍 What is vLLM?
- **Purpose**: A highly optimized inference engine for large language models (LLMs), focusing on performance, scalability, and memory efficiency.
- **Core Features**:
  - Uses asynchronous request processing to handle multiple inputs concurrently.
  - Implements "PagedAttention" to dynamically manage GPU memory during long-sequence generation, reducing wasted space.
  - Supports high-throughput serving via an API (similar to OpenAI's GPT API).
- **How it Works**:  
  1. Loads a model from storage or a framework like PyTorch/TensorRT-LLM.  
  2. Processes requests in batches using GPU parallelism and streaming token generation.  
  3. Optimizes memory allocation by breaking attention computations into smaller chunks ("pages").  

**Usage**:  
```bash
# Example command to serve a model via API (e.g., Llama 2)
vllm server --model meta-llama/Llama-2-7

In [17]:
# stop_ollama('nomic-embed-text:latest')
stop_ollama(OLLAMA_MODEL_NAME)

Stopped Ollama model: deepseek-r1:latest


⠙ 

In [18]:
query = 'Explain About TG glass transition temperature in polymers'
message_2 = [
    {'role':'system', 'content' : system_prompt},
    {'role':'user', 'content':query}
]

for chunk in ollama_model.stream(message_2):
    print(chunk.text,end="",flush=True)


Okay, let's break down the concept of **Tg (Glass Transition Temperature)** in polymers.

## What is Glass Transition?

Imagine heating a solid polymer slowly. At first, nothing much happens until you reach its melting point (*Tm*), where it turns into a liquid melt. However, before reaching *Tm*, there's a crucial change: the material transforms from a **glassy (solid-like)** state to a **rubbery or viscous (soft solid/liquid-like)** state.

This transition isn't sharp like melting; instead, it's more of a gradual softening process as temperature increases. The point where this significant change in physical properties occurs is defined by the **Glass Transition Temperature (*Tg*)**.

### Definition

*   ***Tg*** is the specific temperature at which an amorphous or semi-crystalline polymer transitions from being relatively hard and brittle (glassy state) to soft, viscous, and flexible (rubbery/vitreous liquid state).
*   It marks a change in the polymer's physical properties rather t

In [19]:
stop_ollama(OLLAMA_MODEL_NAME)

Stopped Ollama model: deepseek-r1:latest


In [55]:
template = """You are a helpful assistant that can answer questions and return the reponse in markdown format.
            You are also provided with latest and new context relavent to query. Use the following pieces of context to add more information and answer the question at the end.\n\n{context}\n\nQuestion: {question}\nHelpful Answer:"""
prompt = PromptTemplate.from_template(template)

### RAG Pipeline

In [27]:
qa_chain = RetrievalQA.from_llm(
    llm=ollama_model,
    retriever=retriver,
    prompt=prompt,
    # return_source_documents=True,
    )

In [24]:
for chunk in qa_chain.stream({"query":"what is Tg target in polymer properties, what are the latest advancment in predicting Tg target using Deep Learning, give the model preformance"}):
  print(chunk['result'],end="",flush=True)


Okay, here's a breakdown of the provided text regarding **Tg** (glass transition temperature) as a polymer property target and recent advances using Deep Learning:

## What is Tg Target?

*   **Tg** stands for **Glass Transition Temperature**, which is a key physical property in polymers. It represents the temperature at which an amorphous or semicrystalline polymer transitions from a glassy, rigid state to a rubbery (viscoelastic) state.
*   As a target "in polymer properties," Tg refers to the specific **goal** for designing and developing polymers that can function reliably in high-temperature environments. Polymers with higher Tg values are sought after for applications requiring thermal stability above certain temperatures.

## Latest Advances in Predicting Tg using Deep Learning

*   The context discusses combining unique aspects into a single machine learning framework, including feature space development based on topological descriptors and integration of unsupervised and supe

In [26]:
stop_ollama(model=OLLAMA_MODEL_NAME) 
stop_ollama('nomic-embed-text:latest')

⠙ 

Stopped Ollama model: deepseek-r1:latest


⠙ ⠸ ⠸ ⠼ ⠴ ⠧ 

Stopped Ollama model: nomic-embed-text:latest


⠇ ⠇ 

In [27]:
for chunk in qa_chain.stream({"query":"what is Tg target in polymer properties, what are the latest advancment in predicting Tg target using Deep Learning, give the model preformance"}):
  print(chunk['result'],end="",flush=True)


Okay, let's break down your question based on the provided text:

**1. What is Tg target in polymer properties?**

Tg refers to **Glass Transition Temperature**, a critical property of polymers.

*   It represents the temperature at which a solid polymer transitions from a hard, glassy state (solid-like behavior) into a rubbery, viscous liquid state.
*   At temperatures below Tg, the polymer is brittle and rigid. Above Tg, it becomes softer, more flexible, and viscous rather than elastic like a liquid.
*   **Significance:** It's crucial for determining if a polymer can be used above its glass transition temperature without melting (softening point) or becoming too soft/flowable.

**2. What are the latest advancements in predicting Tg using Deep Learning?**

The context provided focuses on recent work, specifically from a paper published in *ijms* journal (likely 2023 given the PMID prefix). Key points:

*   **Approach:** The advancement involves combining unique aspects into an ML fra

In [28]:
stop_ollama(model=OLLAMA_MODEL_NAME) 
stop_ollama('nomic-embed-text:latest')

⠋ 

Stopped Ollama model: deepseek-r1:latest


⠙ ⠸ ⠸ ⠼ ⠦ ⠦ 

Stopped Ollama model: nomic-embed-text:latest


⠧ 

In [29]:
for chunk in qa_chain.stream({"query":"what is Tg target in polymer properties, what are the latest advancment in predicting Tg target using Deep Learning, give the model preformance"}):
  print(chunk['result'],end="",flush=True)


Okay, based on the provided text, here's a breakdown of Tg and related information:

1.  **What is Tg target in polymer properties?**
    *   From the context, "Tg" refers to the **Glass Transition Temperature (Tg)** of polymers. It represents the temperature at which an amorphous or non-crystalline polymer transitions from a rigid, glassy state to a soft, rubbery state as it is heated.
    *   The text implies that Tg is often used as a target property for designing and developing **sustainable high-Tg polymers** (polymers with targeted high glass transition temperatures), especially in the context of "high-Tg sustainable polymer chemistry" mentioned several times. This suggests focusing on achieving specific, relatively high Tg values using environmentally friendly or novel approaches.

2.  **Latest advancements in predicting Tg target using Deep Learning:**
    *   The text mentions combining unique aspects (like topological descriptors and integrating unsupervised/supervised learn

In [30]:
stop_ollama(model=OLLAMA_MODEL_NAME) 
stop_ollama('nomic-embed-text:latest')

Stopped Ollama model: deepseek-r1:latest
Stopped Ollama model: nomic-embed-text:latest


⠙ 

In [16]:
for chunk in qa_chain.stream({"query":"what is Tg target in polymer properties, what are the latest advancment in predicting Tg target using Deep Learning, give the model preformance"}):
  print(chunk['result'],end="",flush=True)


Okay, let's break down this query based on the provided context:

**Tg Target in Polymer Properties:**

Glass transition temperature (Tg) refers to a specific **target property** related to polymer behavior. It represents the glass transition temperature.

*   **What it is:** Tg is the temperature at which an amorphous or semicrystalline polymer transitions from a hard, glassy state into a rubbery liquid state as it absorbs heat and becomes more flexible.
*   **Why it's important (target):** In polymer science, knowing Tg *a priori* allows engineers to:
    *   Predict the use temperature limits of a polymer.
    *   Select appropriate polymers for specific applications requiring certain mechanical properties at elevated temperatures.
    *   Design polymeric materials with desired thermal stability or flexibility characteristics.

**Latest Advancements in Deep Learning (DL) Tg Prediction:**

The provided context indicates that DL techniques have been applied to predict Tg, often buil

In [19]:
stop_ollama(model=OLLAMA_MODEL_NAME) 
stop_ollama('nomic-embed-text:latest')

⠙ ⠙ 

Stopped Ollama model: deepseek-r1:latest


⠹ ⠸ ⠼ ⠼ ⠦ ⠦ 

Stopped Ollama model: nomic-embed-text:latest


⠧ ⠏ 

In [22]:
for chunk in qa_chain.stream({"query":"what is Tg target in polymer properties, what are the latest advancment in predicting Tg target using Deep Learning, give the model preformance"}):
  print(chunk['result'],end="",flush=True)


Okay, here's the helpful answer based on the provided text:

**Glass Transition Temperature (Tg)**

*   **What is Tg Target?**
    The glass transition temperature (Tg) is a key physical property of polymers. It represents the temperature below which the polymer transitions from a hard and relatively rigid "glassy" state to a softer, rubbery or viscous "leathery" state under an applied load.

**Latest Advancements in Tg Prediction using Deep Learning**

Several recent advancements have been mentioned in the context provided:
1.  **Machine Learning Integration:** There has been significant work combining unique aspects like topological descriptors and integrating unsupervised (e.g., Random Forest for feature importance) and supervised learning approaches.
2.  **Experimental Validation:** Notably, a paper combined machine learning predictions with *experimental validation* of new high-Tg sustainable polymers to verify the model's accuracy.

**Model Performance**

The provided context de

In [25]:
stop_ollama(model=OLLAMA_MODEL_NAME) 
stop_ollama('nomic-embed-text:latest')

Stopped Ollama model: deepseek-r1:latest
Stopped Ollama model: nomic-embed-text:latest


In [30]:
for chunk in qa_chain.stream({"query":"what is Tg target in polymer properties, what are the latest advancment in predicting Tg target using Deep Learning, give the model preformance"}):
  print(chunk['result'],end="",flush=True)


Okay, let's break down the query into its components:

1.  **What is Tg target in polymer properties?**
    *   `Tg` stands for **glass transition temperature** (Tg).
    *   As a target property (`target` or `Tg_target`), it refers to the specific threshold temperature, usually measured in degrees Celsius (°C) or Kelvin (K), at which an amorphous polymer transitions from a hard, glassy state into a rubbery or viscous liquid-like state. It's crucial for determining how well a polymer can withstand heat without becoming too soft or flowing.

2.  **What is the latest advancement in predicting Tg target using Deep Learning?**
    *   The provided context doesn't detail groundbreaking new DL models beyond the specific paper it seems to summarize, but describes their work as an example of recent progress.
    *   It highlights that existing methods often rely on QSPR (Quantitative Structure-Property Relationship) models built with small or congeneric datasets using older techniques like ML

In [31]:
stop_ollama(model=OLLAMA_MODEL_NAME) 
stop_ollama('nomic-embed-text:latest')

Stopped Ollama model: deepseek-r1:latest
Stopped Ollama model: nomic-embed-text:latest


## Quesry Translation

In [26]:
OLLAMA_MODEL_2 = 'llama3.2:latest'

ollama_model_2 = ChatOllama(
    model=OLLAMA_MODEL_2,
    validate_model_on_init=True,
    num_ctx=2046,
    num_predict=6046,
    keep_alive=0
)

### Multi Query

In [86]:
template = """You are an AI language model assistant. Your task is to generate five 
different versions of the given user question to retrieve relevant documents from a vector 
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search. 
Provide these alternative questions separated by newlines.Note:only give qestions noe extra text. Original question: {question}"""
prompt_perspectives = PromptTemplate.from_template(template)


In [87]:
from langchain_core.output_parsers import StrOutputParser

In [88]:
generate_queries = (
    prompt_perspectives | 
    ollama_model_2 |
    StrOutputParser() | 
    (lambda x : x.split("\n"))
)

In [89]:
queries_generatied = generate_queries.invoke({"question":"what is Tg target in polymer properties, what are the latest advancment in predicting Tg target using Deep Learning, give the model preformance"})

In [93]:
print(queries_generatied)

['What are the current polymer properties that have a Tg target?', 'How does Tg target relate to the overall performance of polymers in various applications?', 'What are some recent advances in predicting Tg target using machine learning algorithms?', '', 'How do deep learning models perform in predicting the Tg target for different types of polymers?', 'Can any specific features or characteristics of polymers be used as input to predict their Tg target accurately?', 'Are there any limitations or challenges associated with using deep learning for Tg target prediction?', '', 'What are some potential applications of accurately predicted Tg targets in polymer engineering and materials science?', 'How can the performance of deep learning models for Tg target prediction be evaluated and compared across different datasets?', 'Can combining multiple machine learning algorithms improve the accuracy of Tg target predictions?', 'what is Tg target in polymer properties, what are the latest advanc

In [42]:
from langchain_core.load import dumps, loads

In [119]:

def get_queries(question):
    generate_queries = (
        prompt_perspectives | 
        ollama_model_2 |
        StrOutputParser() | 
        (lambda x : x.split("\n"))
    )
    queries_generatied = generate_queries.invoke({"question":question})
    queries_generatied = [querie for querie in queries_generatied if querie]
    # result = queries_generatied.append(question)
    return queries_generatied

def get_unique_union(documents: list[list]):
    """ Unique union of retrieved docs """
    # Flatten list of lists, and convert each Document to string
    flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]
    # Get unique documents
    unique_docs = list(set(flattened_docs))
    # Return
    return [loads(doc) for doc in unique_docs]


In [120]:
question = "what is Tg target in polymer properties, what are the latest advancment in predicting Tg target using Deep Learning, give the model preformance"


In [121]:
queries = get_queries(question)
queries

['What is the TG target in polymer properties and its significance?',
 'How does the TG target relate to the thermal stability of polymers?',
 'What are the current advancements in predicting the TG target of polymers using machine learning algorithms?',
 'Can deep learning models accurately predict the TG target of polymers based on their molecular structure or other properties?',
 'What is the performance of a recent deep learning model for predicting TG targets in polymers, and how does it compare to traditional methods?']

In [122]:
retrival_chain = get_queries | retriver.map() | get_unique_union

In [123]:
question = "what is Tg target in polymer properties, what are the latest advancment in predicting Tg target using Deep Learning, give the model preformance"

In [124]:
retriver_docs = retriver.invoke(question)

In [125]:
len(retriver_docs)

4

In [ ]:

retrived_docs = retrival_chain.invoke({"question":question})
len(retrived_docs)

In [126]:

retrived_docs = retrival_chain.invoke({"question":question})
len(retrived_docs)

/tmp/ipykernel_21273/1821810733.py:20: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in unique_docs]


20

In [127]:
stop_ollama("nomic-embed-text:latest")

Stopped Ollama model: nomic-embed-text:latest


In [128]:
stop_ollama(OLLAMA_MODEL_2)

Stopped Ollama model: llama3.2:latest


In [129]:
from operator import itemgetter

In [130]:
multi_query_rg = (
    {
        "context": retrival_chain,
        "question": itemgetter("question")
    }
    | prompt
    | ollama_model
    | StrOutputParser()
)

In [131]:

print(multi_query_rg.invoke({"question":question}))

/tmp/ipykernel_21273/1821810733.py:20: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in unique_docs]



## What is Tg (Glass Transition Temperature) in Polymer Properties?

Tg stands for Glass Transition Temperature. It marks the approximate temperature below which a polymer transitions from a relatively soft, flexible, rubbery state to a hard, brittle, glass-like state.

### Key Characteristics:

*   **Thermal Stability:** Polymers with higher Tg values exhibit greater thermal stability and maintain their mechanical integrity at elevated temperatures.
*   **Importance:** It's a critical indicator for many applications (e.g., aerospace, energy, biomedical, electronics) where materials must withstand challenging thermal environments. Most common biopolymers have lower Tg ranges (~55-75°C).
*   **Factors:** Theoretically dependent on chain mobility or free volume, which is influenced by molecular weight, cross-links, side groups, and chain ends.

---

## Latest Advancements in Predicting Polymer Tg using Deep Learning (from the provided context)

Recent advancements leverage machine learn

In [132]:
embedding_model = "nomic-embed-text:latest"
stop_ollama(OLLAMA_MODEL_NAME)
stop_ollama(OLLAMA_MODEL_2)
stop_ollama(embedding_model)

Stopped Ollama model: deepseek-r1:latest


⠙ 

Stopped Ollama model: llama3.2:latest
Stopped Ollama model: nomic-embed-text:latest
